<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day09_decision_tree_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 9: Decision Tree (Basics)
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Supervised learning, Decision Trees

> **Goal of today:** Understand how a tree splits the data, draw and read a small tree, and see overfitting happen.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load data. bad = 1 is the positive class
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
X = df.drop(columns=['class'])
y = df['class'].map({'good': 0, 'bad': 1}).astype(int)

# Stratified 80/20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing fitted on train only
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

preprocess = ColumnTransformer(
    [('num', StandardScaler(), num_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    sparse_threshold=0
)
preprocess.set_output(transform="pandas")

X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)
print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)


def evaluate(name, y_true, pred):
    """Return the four main metrics for the positive class (bad = 1)."""
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_true, pred), 3),
        'precision': round(precision_score(y_true, pred, zero_division=0), 3),
        'recall': round(recall_score(y_true, pred, zero_division=0), 3),
        'f1': round(f1_score(y_true, pred, zero_division=0), 3),
    }

Train: (800, 61) | Test: (200, 61)


## 2. Gini Impurity by Hand

**Gini = 1 - (p_good² + p_bad²)**

| Node | Gini |
|---|---|
| all one class | 0 (pure) |
| 70 / 30 | 0.42 |
| 50 / 50 | 0.50 (most mixed) |

We compute it for our full training set, then for one example split.

In [2]:
def gini(labels):
    """Gini impurity of a set of 0/1 labels."""
    p_bad = np.mean(labels)        # fraction of 1s (bad)
    p_good = 1 - p_bad
    return 1 - (p_good**2 + p_bad**2)

print("Gini of the full training set:", round(gini(y_train), 3))

# Try one example split: customers with duration <= 18 months vs more than 18
raw_train = X_train.copy()
left = y_train[raw_train['duration'] <= 18]
right = y_train[raw_train['duration'] > 18]

weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(y_train)
print("Left size:", len(left), "| Gini:", round(gini(left), 3))
print("Right size:", len(right), "| Gini:", round(gini(right), 3))
print("Weighted Gini after the split:", round(weighted, 3))
print("Gain:", round(gini(y_train) - weighted, 3))

Gini of the full training set: 0.42
Left size: 437 | Gini: 0.365
Right size: 363 | Gini: 0.467
Weighted Gini after the split: 0.411
Gain: 0.009


## 3. A Tree With No Limits

With default settings the tree keeps splitting until every leaf is pure. Watch the gap between **train** and **test** scores.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# No limits: the tree can grow as deep as it wants
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train_p, y_train)

print("Depth:", tree_full.get_depth(), "| Leaves:", tree_full.get_n_leaves())
print("Train accuracy:", round(tree_full.score(X_train_p, y_train), 3))
print("Test accuracy :", round(tree_full.score(X_test_p, y_test), 3))

## 4. A Small Tree We Can Read

We limit the depth to 3. Each box in the drawing shows:

| Line in the box | Meaning |
|---|---|
| first line | the question (left branch = yes, right branch = no) |
| gini | impurity of that node |
| samples | rows that reach this node |
| value | [number of good, number of bad] |
| class | majority class, which is the prediction |

In [3]:
from sklearn.tree import plot_tree

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train_p, y_train)

plt.figure(figsize=(18, 8))
plot_tree(
    tree3,
    feature_names=X_train_p.columns,
    class_names=['good', 'bad'],
    filled=True,         # color shows the majority class
    rounded=True,
    fontsize=8,
)
plt.show()

NameError: name 'DecisionTreeClassifier' is not defined